In [59]:
# Updated imports
!pip install qiskit-ionq
import random
import hashlib
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister, transpile
from qiskit_ionq import IonQProvider
from typing import List, Tuple
from qiskit_aer import AerSimulator
import numpy as np
import os

# Initialize IonQ provider
# Set IONQ_API_KEY environment variable or pass directly
provider = IonQProvider("uPshzBiQ04JDnTJBbcEmoCvX1Sf0MToK")


# UNCOMMENT BELOW IF YOU WANT TO CHECK WHICH BACKENDS ARE AVAILABLE RIGHT NOW
# def test_backend_connection(backend_name): # Checks which backends are available right now
#     """Test if a specific backend works"""
#     try:
#         backend = provider.get_backend(backend_name)
#         print(f"✅ Backend '{backend_name}' exists")
        
#         # Test with minimal circuit
#         from qiskit import QuantumCircuit
#         qc = QuantumCircuit(1, 1)
#         qc.measure(0, 0)
        
#         print(f"   Testing with minimal circuit...")
#         job = backend.run(qc, shots=1)
#         result = job.result()
#         print(f"   ✅ '{backend_name}' works successfully!")
#         return True
        
#     except Exception as e:
#         print(f"   ❌ '{backend_name}' failed: {e}")
#         return False
# # Test all available backends
# print("Testing all available backends:")
# available_backends = provider.backends()
# for backend in available_backends:
#     test_backend_connection(backend.name)

# UNCOMMENT BELOW BASED ON WHCIH BACKEND YOU ARE WORKING WITH

# backend = provider.get_backend("ionq_simulator")  # or "ionq_qpu" for real hardware
# flag = 0 # flag to tell when it is in simulator vs. actual quantum hardware, 0 means simulator

backend = provider.get_backend("ionq_qpu.forte-1")  # or "ionq_qpu"/"ionq_qpu.forte-1" for real hardware
flag = 1 # flag to tell when it is in simulator vs. actual quantum hardware, 1 means quantum hardware


# backend = AerSimulator()
# flag = 0 # flag to tell when it is in simulator vs. actual quantum hardware, 0 means AerSimulator



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [58]:
def random_bits(n: int) -> List[int]:
    return [random.randint(0, 1) for _ in range(n)]

def random_bases(n: int) -> List[int]:
    return [random.randint(0, 1) for _ in range(n)]

def prepare_batch_circuit(bits: List[int], bases: List[int]) -> QuantumCircuit:
    """Create a single circuit with multiple qubits for batch processing"""
    n = len(bits)
    qc = QuantumCircuit(n, n) # creates a quantum circuit with n qubits and n classical bits
    
    for i, (bit, basis) in enumerate(zip(bits, bases)):
        if bit == 1: # convert qubit to |1> by applying an X gate, this is necessary because by default qubits start in the |0> state
            qc.x(i)
        if basis == 1: # if basis is X, apply H to convert from |0> to |+> or from |1> to |->
            qc.h(i)
    
    return qc

def add_noise_to_circuit(qc: QuantumCircuit, noise_level: float = 0.05, noise_type: str = "depolarizing") -> QuantumCircuit:
    """Applies simulated noise to each qubit in the circuit."""
    noisy_qc = qc.copy()

    for i in range(noisy_qc.num_qubits):
        if random.random() < noise_level:
            if noise_type == "bitflip":
                noisy_qc.x(i)
            elif noise_type == "phaseflip":
                noisy_qc.z(i)
            elif noise_type == "depolarizing":
                gate = random.choice(['x', 'y', 'z'])
                getattr(noisy_qc, gate)(i)

    return noisy_qc


def measure_batch_circuit(base_qc: QuantumCircuit, bases: List[int]) -> QuantumCircuit:
    """Add measurement operations for Bob's bases"""
    n = base_qc.num_qubits
    qc = base_qc.copy()
    
    for i, basis in enumerate(bases):
        if basis == 1:
            qc.h(i) # applies hadamard gate to qubit at index i
        qc.measure(i, i) # measures qubit in index i and stores it as a classical bit in index i
    
    return qc

def run_circuit_get_bits(qc: QuantumCircuit, shots: int = 1) -> List[int]:
    """Run circuit on IonQ and return measured bits"""
    # Transpile for IonQ backend
    t_qc = transpile(qc, backend=backend)
    
    # Submit job
    job = backend.run(t_qc, shots=shots)
    result = job.result()
    
    # Get counts, returns a dictionary with the key equal to the result and the value equal to the number of times it ocurred
    counts = result.get_counts()
    
    # For single shot, return the measured outcome
    if shots == 1:
        # Convert from binary string to list of bits
        outcome = list(counts.keys())[0]  # Get the single outcome
        return [int(bit) for bit in outcome[::-1]]  # Reverse for Qiskit ordering
    
    # For multiple shots, return most frequent outcome
    most_frequent = max(counts, key=counts.get) # finds which key has more occurences
    return [int(bit) for bit in most_frequent[::-1]]

def eve_intercept_resend_batch(alice_bits: List[int], alice_bases: List[int], eve_bases: List[int]) -> QuantumCircuit:
    """Eve intercepts and resends using batch processing"""
    # Eve measures in her bases
    alice_circuit = prepare_batch_circuit(alice_bits, alice_bases)
    eve_measure_circuit = measure_batch_circuit(alice_circuit, eve_bases)
    eve_measurements = run_circuit_get_bits(eve_measure_circuit)
    
    # Eve re-prepares qubits in her measured states
    return prepare_batch_circuit(eve_measurements, eve_bases)

def sift_key(alice_bits: List[int], alice_bases: List[int], bob_bits: List[int], bob_bases: List[int]) -> Tuple[List[int], List[int], List[int]]:
    """Return (sifted_alice, sifted_bob, indices_kept) where we keep positions with matching bases."""
    sifted_a = []
    sifted_b = []
    indices = []
    for i, (abits, abases, bbits, bbases) in enumerate(zip(alice_bits, alice_bases, bob_bits, bob_bases)):
        if alice_bases[i] == bob_bases[i]:
            sifted_a.append(alice_bits[i])
            sifted_b.append(bob_bits[i])
            indices.append(i)
    return sifted_a, sifted_b, indices

def error_rate(a_bits: List[int], b_bits: List[int]) -> float:
    if not a_bits:
        return 0.0
    mismatches = sum(x != y for x, y in zip(a_bits, b_bits))
    return mismatches / len(a_bits)

def cascade_error_correction(alice_key: List[int], bob_key: List[int], num_passes: int = 4) -> Tuple[List[int], List[int], int]:
    """Simplified Cascade protocol for BB84 error correction."""
    corrected_bob = bob_key.copy()
    total_corrections = 0

    key_length = len(alice_key)
    if key_length == 0:
        return alice_key, bob_key, 0

    # Cascade has multiple passes with different block sizes
    for p in range(num_passes):
        block_size = max(1, key_length // (2 ** (p + 1)))  # progressively smaller blocks
        indices = list(range(0, key_length, block_size))
        random.shuffle(indices)  # random permutation for each pass

        for start in indices:
            end = min(start + block_size, key_length)
            a_block = alice_key[start:end]
            b_block = corrected_bob[start:end]

            # Compare parities
            if sum(a_block) % 2 != sum(b_block) % 2:
                # Binary search for the first mismatch
                left, right = 0, len(a_block) - 1
                while left < right:
                    mid = (left + right) // 2
                    if (sum(a_block[:mid+1]) % 2) != (sum(b_block[:mid+1]) % 2):
                        right = mid
                    else:
                        left = mid + 1
                # Flip the found bit in Bob's key
                corrected_bob[start + left] ^= 1
                total_corrections += 1

    return alice_key, corrected_bob, total_corrections

def privacy_amplification(shared_key: List[int], final_length: int = None) -> List[int]:
    """
    Perform privacy amplification using SHA-256 hashing.
    Compresses the key to a smaller length to remove Eve's possible knowledge.
    """
    # Convert bit list to string
    bitstring = ''.join(map(str, shared_key))
    
    # Hash with SHA-256
    hashed = hashlib.sha256(bitstring.encode()).hexdigest()
    
    # Convert hex hash to bits
    hashed_bits = bin(int(hashed, 16))[2:].zfill(256)
    
    # If no target length given, make final key half the size
    if final_length is None:
        final_length = len(shared_key) // 2
    final_bits = [int(b) for b in hashed_bits[:final_length]]
    
    return final_bits


def create_full_bb84_circuit(alice_bits: List[int], alice_bases: List[int], bob_bases: List[int]) -> QuantumCircuit:
    """Create a complete BB84 circuit for visualization (like the example)"""
    n = len(alice_bits)
    qc = QuantumCircuit(n, n)
    
    # Alice's encoding
    for i in range(n):
        if alice_bases[i] == 0:  # Z-basis
            if alice_bits[i] == 1:
                qc.x(i)
        else:  # X-basis
            if alice_bits[i] == 0:
                qc.h(i)
            else:
                qc.x(i)
                qc.h(i)
    
    qc.barrier()  # Visual separation
    
    # Bob's measurement
    for i in range(n):
        if bob_bases[i] == 1:  # X-basis measurement
            qc.h(i)
    
    qc.measure(range(n), range(n))
    return qc

# def generate_ldpc_matrix(n_bits: int, n_parity: int) -> np.ndarray:
#     """
#     Generate a simple random LDPC parity-check matrix H with n_parity rows
#     and n_bits columns. This is not optimized for coding performance,
#     just for demonstration.
#     """
#     H = np.zeros((n_parity, n_bits), dtype=int)
#     for i in range(n_parity):
#         ones_positions = np.random.choice(n_bits, size=max(2, n_bits // 8), replace=False)
#         H[i, ones_positions] = 1
#     return H

# def ldpc_encode(key: List[int], H: np.ndarray) -> List[int]:
#     """
#     Simple systematic encoding for demonstration: returns key + parity bits
#     Here we just append parity bits computed from H * key
#     """
#     key_bits = np.array(key)
#     parity_bits = (H @ key_bits) % 2
#     return np.concatenate([key_bits, parity_bits]).tolist()

# def ldpc_decode(received: List[int], H: np.ndarray, max_iters: int = 10) -> List[int]:
#     """
#     Bit-flip iterative LDPC decoder
#     received: received bits including parity
#     H: parity-check matrix
#     Returns the corrected key (first n_bits)
#     """
#     r = np.array(received)
#     n_bits = H.shape[1]
#     for _ in range(max_iters):
#         syndrome = (H @ r[:n_bits]) % 2
#         if np.all(syndrome == 0):
#             break  # No errors detected
#         # Flip bits involved in the largest number of failing parity checks
#         flip_count = H.T @ syndrome
#         flip_index = np.argmax(flip_count)
#         r[flip_index] ^= 1
#     return r[:n_bits].tolist()

# def ldpc_error_correction(alice_key: List[int], bob_key: List[int], n_parity: int = None):
#     """
#     Apply LDPC error correction to Bob's key to match Alice's key.
#     """
#     n_bits = len(alice_key)
#     if n_parity is None:
#         n_parity = max(2, n_bits // 4)
    
#     H = generate_ldpc_matrix(n_bits, n_parity)
#     encoded_alice = ldpc_encode(alice_key, H)
#     encoded_bob = ldpc_encode(bob_key, H)  # In practice, Bob has errors
#     corrected_bob = ldpc_decode(encoded_bob, H)
    
#     total_corrections = sum(a != b for a, b in zip(alice_key, corrected_bob))
#     return alice_key, corrected_bob, total_corrections

def generate_ldpc_matrix(n_bits: int, n_parity: int) -> np.ndarray:
    """
    Generate a small random LDPC parity-check matrix H.
    For tiny keys (n_bits <= 16), we just make each row have 2-3 ones.
    """
    H = np.zeros((n_parity, n_bits), dtype=int)
    for i in range(n_parity):
        ones_positions = np.random.choice(n_bits, size=min(3, n_bits), replace=False)
        H[i, ones_positions] = 1
    return H

def ldpc_syndrome(key: List[int], H: np.ndarray) -> np.ndarray:
    """Compute the syndrome: H * key (mod 2)"""
    key_vec = np.array(key)
    return (H @ key_vec) % 2

def ldpc_decode_bob(bob_key: List[int], H: np.ndarray, syndrome: np.ndarray, max_iters: int = 10) -> List[int]:
    """
    Iterative bit-flip decoder using only syndrome.
    Bob flips bits involved in most unsatisfied parity checks.
    """
    r = np.array(bob_key)
    n_bits = len(r)
    for _ in range(max_iters):
        synd = (H @ r) % 2
        if np.all(synd == 0):
            break  # all parity checks satisfied
        flip_count = H.T @ synd
        if np.all(flip_count == 0):
            break
        # Flip the bit involved in the largest number of unsatisfied checks
        flip_index = np.argmax(flip_count)
        r[flip_index] ^= 1
    return r.tolist()

def ldpc_error_correction(alice_key: List[int], bob_key: List[int], n_parity: int = None):
    """
    Apply LDPC error correction: 
    Alice sends the syndrome to Bob; Bob decodes using his noisy key.
    Returns corrected Bob key and total corrections.
    """
    n_bits = len(alice_key)
    if n_parity is None:
        n_parity = max(2, n_bits // 2)  # small number of parity bits

    H = generate_ldpc_matrix(n_bits, n_parity)
    syndrome = ldpc_syndrome(alice_key, H)
    corrected_bob = ldpc_decode_bob(bob_key, H, syndrome)

    total_corrections = sum(a != b for a, b in zip(alice_key, corrected_bob))
    return alice_key, corrected_bob, total_corrections


def run_bb84(n: int = 8, with_eve: bool = False, verbose: bool = True, print_circuits: bool = False, 
             noise_level: float = 0.0, noise_type: str = "depolarizing", ec: str = "none", privacy_amp: bool = False):
    
    """Run BB84 - note smaller n due to hardware limitations"""
    if n > 11 and flag:  # IonQ currently has limited qubits
        print(f"Warning: Reducing n from {n} to 8 due to hardware limitations")
        n = 8
    
    # Alice chooses bits and bases
    alice_bits = random_bits(n)
    alice_bases = random_bases(n)

    # Prepare circuits
    if with_eve:
        eve_bases = random_bases(n)
        # Eve intercepts and resends
        # Let's create a simple Eve simulation
        print("\n" + "="*50)
        print("EVE INTERCEPT-RESEND ATTACK:")
        print("="*50)
        print(f"Eve's bases: {eve_bases}")
        # In batch mode, Eve would need separate handling
        forwarded_circuit = eve_intercept_resend_batch(alice_bits, alice_bases, eve_bases)
    else:
        # No Eve - use Alice's original preparation
        forwarded_circuit = prepare_batch_circuit(alice_bits, alice_bases)

    if noise_level > 0:
        print("\n" + "="*50)
        print("ADDING SIMULATED NOISE TO CHANNEL")
        print("="*50)
        print(f"Noise level: {noise_level:.2f}, Type: {noise_type}")
        forwarded_circuit = add_noise_to_circuit(forwarded_circuit, noise_level, noise_type)


    # Bob chooses measurement bases and measures
    bob_bases = random_bases(n)
    bob_measure_circuit = measure_batch_circuit(forwarded_circuit, bob_bases)

    if print_circuits:
        full_circuit = create_full_bb84_circuit(alice_bits, alice_bases, bob_bases)
        print("\n" + "="*50)
        print("COMPLETE BB84 CIRCUIT DIAGRAM:")
        print("="*50)
        print(full_circuit)
        print("\n")
        print(f"Total qubits: {full_circuit.num_qubits}")
        print(f"Circuit depth: {full_circuit.depth()}")
        print(f"Gate counts: {dict(full_circuit.count_ops())}")
        print("\n")

    # Run on desired backend
    bob_bits = run_circuit_get_bits(bob_measure_circuit)

    # Sift keys
    sifted_a, sifted_b, indices = sift_key(alice_bits, alice_bases, bob_bits, bob_bases)
    
    # Calculate error rate
    err = error_rate(sifted_a, sifted_b)

    # performs Cascade Error Correction
    if ec == 'cascade':
        sifted_a_corr, sifted_b_corr, corrections = cascade_error_correction(sifted_a, sifted_b, num_passes = 4)
        err_after = error_rate(sifted_a_corr, sifted_b_corr)

    if ec == 'ldpc':  # You can rename this flag to 'error_correction' to include LDPC too
        sifted_a_corr, sifted_b_corr, corrections = ldpc_error_correction(sifted_a, sifted_b)
        err_after = error_rate(sifted_a_corr, sifted_b_corr)


    if privacy_amp:
        final_key = privacy_amplification(sifted_a_corr)

    if verbose:
        print(f"Using backend: {backend.name}")
        print(f"Alice bits:      {alice_bits}")
        print(f"Alice bases:     {alice_bases}")
        print(f"Bob bits:        {bob_bits}")
        print(f"Bob bases:       {bob_bases}")
        print(f"Sifted indices:  {indices}")
        print(f"Sifted Alice:    {sifted_a}")
        print(f"Sifted Bob:      {sifted_b}")
        if ec == "cascade":
            print("\n")
            print("With Cascade:")
            print(f"Error rate before Cascade: {err}")
            print(f"Total bit corrections: {corrections}")
            print(f"Error rate after Cascade:  {err_after}")
        elif ec == "ldpc":
            print("\n")
            print("With LDPC:")
            print(f"Error rate before LDPC: {err}")
            print(f"Total bit corrections: {corrections}")
            print(f"Error rate after LDPC:  {err_after}")
        else:
            print(f"Error rate:      {err}")
        print(f"Key length:      {len(sifted_a)}")
        if privacy_amp:
            print("\n")
            print("With Privacy Amplification:")
            print(f"Final key after Privacy Amplification: {final_key}")
            print(f"Final key length after Privacy Amplification: {len(final_key)}")
        

    return {
        'alice_bits': alice_bits,
        'alice_bases': alice_bases,
        'bob_bits': bob_bits,
        'bob_bases': bob_bases,
        'sifted_alice': sifted_a,
        'sifted_bob': sifted_b,
        'sifted_indices': indices,
        'error_rate': err,
        'qber': err,
        # 'error_rate_after': err_after,
    }

# Test
print("Testing BB84:")
result = run_bb84(n = 8, with_eve = False, verbose = True, print_circuits = True, noise_level = 0.0, noise_type = "depolarizing", ec ='none', privacy_amp = False)

Testing BB84:

COMPLETE BB84 CIRCUIT DIAGRAM:
     ┌───┐      ░      ┌─┐                     
q_0: ┤ H ├──────░──────┤M├─────────────────────
     ├───┤┌───┐ ░      └╥┘┌─┐                  
q_1: ┤ X ├┤ H ├─░───────╫─┤M├──────────────────
     └───┘└───┘ ░ ┌───┐ ║ └╥┘         ┌─┐      
q_2: ───────────░─┤ H ├─╫──╫──────────┤M├──────
     ┌───┐      ░ └───┘ ║  ║ ┌─┐      └╥┘      
q_3: ┤ X ├──────░───────╫──╫─┤M├───────╫───────
     ├───┤┌───┐ ░       ║  ║ └╥┘┌─┐    ║       
q_4: ┤ X ├┤ H ├─░───────╫──╫──╫─┤M├────╫───────
     ├───┤├───┤ ░       ║  ║  ║ └╥┘┌─┐ ║       
q_5: ┤ X ├┤ H ├─░───────╫──╫──╫──╫─┤M├─╫───────
     ├───┤└───┘ ░ ┌───┐ ║  ║  ║  ║ └╥┘ ║ ┌─┐   
q_6: ┤ X ├──────░─┤ H ├─╫──╫──╫──╫──╫──╫─┤M├───
     ├───┤      ░ ├───┤ ║  ║  ║  ║  ║  ║ └╥┘┌─┐
q_7: ┤ X ├──────░─┤ H ├─╫──╫──╫──╫──╫──╫──╫─┤M├
     └───┘      ░ └───┘ ║  ║  ║  ║  ║  ║  ║ └╥┘
c: 8/═══════════════════╩══╩══╩══╩══╩══╩══╩══╩═
                        0  1  3  4  5  2  6  7 


Total qubits: 8
Circuit depth: 4
Gate co

Machine Learning Implementation

In [25]:
!pip install pyldpc
import numpy as np
import numpy as np
import pandas as pd
import random
!pip install rich
from rich.progress import track
from pyldpc import make_ldpc, encode, decode, get_message




  Using cached pyldpc-0.7.9.tar.gz (1.1 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... error
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [23 lines of output]
      Traceback (most recent call last):
        File "/Users/evan/Documents/UMD Classes/FIRE/FIRE298/quantum_projects/bb84-env/lib/python3.13/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 389, in <module>
          main()
          ~~~~^^
        File "/Users/evan/Documents/UMD Classes/FIRE/FIRE298/quantum_projects/bb84-env/lib/python3.13/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ~~~~^^^^^^^^^^^^^^^^^^^^^^^^
        File "/Users/evan/Documents/UMD Classes/FIRE/FIRE298/quantum_projects/bb84-env/lib/python3.13/site-packages/pip/_vendor

ModuleNotFoundError: No module named 'pyldpc'

In [27]:
# def apply_ec_method(alice_key, bob_key, method="cascade", block_size=8):
#     """
#     Apply real (or simplified) error correction algorithms to align Bob's key with Alice's.
#     Returns corrected Bob key, final key length, and residual QBER.
#     """
#     if len(alice_key) == 0:
#         return {"method": method, "final_key_length": 0, "residual_qber": np.nan}

#     alice_key = np.array(alice_key)
#     bob_key = np.array(bob_key)
#     n = len(alice_key)

#     # ------------- Parity Check (baseline) -------------
#     if method == "parity":
#         corrected = bob_key.copy()
#         for i in range(0, n, block_size):
#             block_a = alice_key[i:i + block_size]
#             block_b = corrected[i:i + block_size]
#             if len(block_a) == 0:
#                 continue
#             # Compare parity
#             if np.sum(block_a) % 2 != np.sum(block_b) % 2:
#                 # flip a random bit to "fix" parity (simplified)
#                 corrected[i] ^= 1
#         residual_qber = np.mean(alice_key != corrected)
#         return {
#             "method": method,
#             "final_key_length": len(corrected),
#             "residual_qber": residual_qber,
#         }

#     # ------------- Cascade Algorithm -------------
#     elif method == "cascade":
#         corrected = bob_key.copy()
#         num_passes = 4  # typical number of cascade passes
#         block_size = max(4, n // 16)

#         for p in range(num_passes):
#             # Divide key into random blocks
#             indices = np.arange(n)
#             np.random.shuffle(indices)
#             for i in range(0, n, block_size):
#                 block_idx = indices[i:i + block_size]
#                 block_a = alice_key[block_idx]
#                 block_b = corrected[block_idx]

#                 # Compare parities
#                 if np.sum(block_a) % 2 != np.sum(block_b) % 2:
#                     # Binary search for error position
#                     low, high = 0, len(block_idx) - 1
#                     while low < high:
#                         mid = (low + high) // 2
#                         if (np.sum(block_a[:mid+1]) % 2) != (np.sum(block_b[:mid+1]) % 2):
#                             high = mid
#                         else:
#                             low = mid + 1
#                     # Flip bit at detected position
#                     corrected[block_idx[low]] ^= 1

#             # Shuffle key for next pass (simulating interleaving)
#             np.random.shuffle(corrected)

#         residual_qber = np.mean(alice_key != corrected)
#         return {
#             "method": method,
#             "final_key_length": len(corrected),
#             "residual_qber": residual_qber,
#         }

#     # ------------- LDPC Algorithm (via pyldpc) -------------
#     elif method == "ldpc":
#         # Build LDPC matrices
#         n_var = n
#         d_v, d_c = 3, 6  # variable/check node degrees
#         H, G = make_ldpc(n_var, d_v, d_c, systematic=True, sparse=True)
        
#         # Encode Alice’s key
#         x = encode(G, alice_key % 2)
        
#         # Add noise to simulate Bob’s key mismatch
#         noisy = (x + (alice_key != bob_key).astype(int)) % 2

#         # Decode to correct errors
#         decoded = decode(H, noisy, maxiter=50)
#         message = get_message(G, decoded)

#         # Pad or trim to original length
#         corrected = np.resize(message, n)
#         residual_qber = np.mean(alice_key != corrected)

#         return {
#             "method": method,
#             "final_key_length": len(corrected),
#             "residual_qber": residual_qber,
#         }

#     else:
#         raise ValueError(f"Unknown EC method: {method}")

import numpy as np

def apply_ec_method(alice_key, bob_key, method="cascade", block_size=8):
    """
    Apply error correction algorithms to align Bob's key with Alice's.
    
    Supported methods:
    - "parity": simple parity check per block (baseline)
    - "cascade": iterative parity-based correction with binary search

    Returns:
        dict with keys:
        - method: which EC method was applied
        - final_key_length: length of corrected key
        - residual_qber: QBER after correction
    """
    if len(alice_key) == 0:
        return {"method": method, "final_key_length": 0, "residual_qber": np.nan}

    # Convert to numpy arrays for convenience
    alice_key = np.array(alice_key)
    bob_key = np.array(bob_key)
    n = len(alice_key)

    # ------------------- Parity Check -------------------
    if method == "parity":
        corrected = bob_key.copy()
        for i in range(0, n, block_size):
            block_a = alice_key[i:i + block_size]
            block_b = corrected[i:i + block_size]
            if len(block_a) == 0:
                continue  # skip empty block

            # Compare parity
            if np.sum(block_a) % 2 != np.sum(block_b) % 2:
                # Flip first bit as a simple fix
                corrected[i] ^= 1

        residual_qber = np.mean(alice_key != corrected)
        return {
            "method": method,
            "final_key_length": len(corrected),
            "residual_qber": residual_qber,
        }

    # ------------------- Cascade Algorithm -------------------
    elif method == "cascade":
        corrected = bob_key.copy()
        num_passes = 3  # fewer passes for faster simulation
        block_size = max(4, n // 16)  # initial block size

        for p in range(num_passes):
            # Shuffle indices for interleaving (simulates random block selection)
            indices = np.arange(n)
            np.random.shuffle(indices)

            for i in range(0, n, block_size):
                block_idx = indices[i:i + block_size]
                if len(block_idx) == 0:
                    continue  # skip empty block

                block_a = alice_key[block_idx]
                block_b = corrected[block_idx]

                # Check parity
                if np.sum(block_a) % 2 != np.sum(block_b) % 2:
                    # Binary search for mismatch with max iteration safeguard
                    low, high = 0, len(block_idx) - 1
                    max_iter = 50
                    iter_count = 0

                    while low < high and iter_count < max_iter:
                        mid = (low + high) // 2
                        if (np.sum(block_a[:mid + 1]) % 2) != (np.sum(block_b[:mid + 1]) % 2):
                            high = mid
                        else:
                            low = mid + 1
                        iter_count += 1

                    # Flip the detected bit (or fallback)
                    if iter_count == max_iter:
                        corrected[block_idx[0]] ^= 1
                    else:
                        corrected[block_idx[low]] ^= 1

        # Residual QBER after Cascade
        residual_qber = np.mean(alice_key != corrected)
        return {
            "method": method,
            "final_key_length": len(corrected),
            "residual_qber": residual_qber,
        }

    else:
        raise ValueError(f"Unknown EC method: {method}")


def build_dataset(trials=10, n=128):
    noise_types = ["bitflip", "depolarizing"]
    noise_levels = np.linspace(0.01, 0.1, 4)
    ec_methods = ["parity", "cascade", "ldpc"]

    records = []
    for t in track(range(trials), description="Running BB84 + EC simulations..."):
        for noise_type in noise_types:
            for nl in noise_levels:
                # Run one BB84 simulation
                result = run_bb84(n=n, with_eve=False, noise_level=nl, noise_type=noise_type)

                # for method in ["parity", "cascade", "ldpc"]:
                for method in ["parity", "cascade"]:
                    ec_result = apply_ec_method(result["alice_key"], result["bob_key"], method)
                    records.append({
                        "trial": t,
                        "noise_type": noise_type,
                        "noise_level": nl,
                        "message_length": n,
                        "initial_qber": result["qber"],
                        "ec_method": method,
                        "final_key_length": ec_result["final_key_length"],
                        "residual_qber": ec_result["residual_qber"],
                    })


    df = pd.DataFrame(records)
    return df

# ----------------------------------------------------------
# 4️⃣ Run and save dataset
# ----------------------------------------------------------
print("Running test BB84 + EC simulation...")
df_results = build_dataset(trials=1, n=64)
df_results.to_csv("bb84_ec_dataset.csv", index=False)
print("✅ Dataset saved as bb84_ec_dataset.csv")
print(df_results.head())

Running test BB84 + EC simulation...


==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
==================================================
======================

RecursionError: maximum recursion depth exceeded

In [17]:
def build_dataset(trials=50, n=8):
    """
    Run BB84 multiple times to collect statistics.
    Vary noise levels and noise types to see how QBER behaves.
    This dataset can later train a model to choose the best error correction algorithm.
    """
    noise_types = ["bitflip", "phaseflip", "depolarizing"]
    noise_levels = np.linspace(0, 0.3, 7)  # from 0% to 30% noise
    records = []

    for t in range(trials):
        for noise_type in noise_types:
            for nl in noise_levels:
                result = run_bb84(n=n, with_eve=False, noise_level=nl, noise_type=noise_type)
                records.append(result)
                print(f"Trial {t+1}/{trials}, noise={nl:.2f}, type={noise_type}, QBER={result['qber']:.2f}")

    # Convert results into a DataFrame for easy analysis or ML
    df = pd.DataFrame(records)
    print("\n✅ Dataset built successfully.")
    print(df.head())
    return df


print("Running test BB84 simulation...")
df_results = build_dataset(trials=5, n=6)

# Save dataset to file for future ML use
df_results.to_csv("bb84_noise_dataset.csv", index=False)
print("\nSaved dataset to bb84_noise_dataset.csv")

Running test BB84 simulation...
Using backend: ionq_simulator
Alice bits:      [0, 0, 1, 1, 0, 1]
Alice bases:     [0, 1, 1, 1, 1, 0]
Bob bits:        [0, 0, 1, 1, 0, 1]
Bob bases:       [1, 1, 1, 0, 1, 0]
Sifted indices:  [1, 2, 4, 5]
Sifted Alice:    [0, 1, 0, 1]
Sifted Bob:      [0, 1, 0, 1]
Error rate:      0.0
Key length:      4
Trial 1/5, noise=0.00, type=bitflip, QBER=0.00

ADDING SIMULATED NOISE TO CHANNEL
Noise level: 0.05, Type: bitflip
Using backend: ionq_simulator
Alice bits:      [0, 1, 0, 1, 1, 0]
Alice bases:     [1, 0, 1, 1, 1, 1]
Bob bits:        [1, 1, 0, 0, 0, 1]
Bob bases:       [0, 0, 1, 0, 0, 0]
Sifted indices:  [1, 2]
Sifted Alice:    [1, 0]
Sifted Bob:      [1, 0]
Error rate:      0.0
Key length:      2
Trial 1/5, noise=0.05, type=bitflip, QBER=0.00

ADDING SIMULATED NOISE TO CHANNEL
Noise level: 0.10, Type: bitflip
Using backend: ionq_simulator
Alice bits:      [1, 0, 1, 1, 0, 0]
Alice bases:     [1, 1, 0, 0, 1, 1]
Bob bits:        [1, 1, 0, 1, 1, 0]
Bob bases:

KeyboardInterrupt: 

In [4]:
!pip install scikit-learn pandas # machine learning library
!pip install pyldpc #LDPC library

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 17.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 24.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [pandas]2m5/6 [pandas]learn]

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 11.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... error
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [23 lines of output]
      Traceback (most recent call last):
        File "/Users/evan/Documents/UMD Classes/FIRE/FIRE298/quantum_projects/bb84-env/lib/python3.13/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 389, in <module>
          main()
          ~~~~^^
        File "/Users/evan/Documents/UMD Classes/F

In [6]:
import pandas as pd
from collections import deque

def qber_per_basis(alice_bits, alice_bases, bob_bits, bob_bases):
    # compute QBER separately for basis 0 and 1 (Z and X)
    z_idx = [i for i,(ab,bb) in enumerate(zip(alice_bases, bob_bases)) if ab==bb==0]
    x_idx = [i for i,(ab,bb) in enumerate(zip(alice_bases, bob_bases)) if ab==bb==1]
    def rate(idx): # iterates through each list of matched bases and checks counts the rate of mismatches
        if not idx: return 0.0
        mismatches = sum(alice_bits[i]!=bob_bits[i] for i in idx)
        return mismatches/len(idx)
    return rate(z_idx), rate(x_idx)

def burstiness(error_vector):
    # error_vector: list of 0/1 where 1 means mismatch in sifted positions
    if not error_vector: return 0.0
    runs = []
    cur = 0
    for e in error_vector:
        if e==1:
            cur += 1
        else:
            if cur>0:
                runs.append(cur)
                cur = 0
    if cur>0: runs.append(cur)
    if not runs: return 0.0
    return float(sum(runs))/len(runs)  # mean run length

def error_autocorr(error_vector, lag=1):
    n = len(error_vector)
    if n <= lag: return 0.0
    mean = sum(error_vector)/n
    num = sum((error_vector[i]-mean)*(error_vector[i+lag]-mean) for i in range(n-lag))
    den = sum((e-mean)**2 for e in error_vector) or 1.0
    return num/den

def extract_features(run_result: dict):
    """
    run_result: dict returned by run_bb84 (or construct similarly)
    returns: dict of features for ML: qber, qber_z, qber_x, block_length, burstiness, autocorr, noise params
    """
    # some keys expected in run_result: 'sifted_alice', 'sifted_bob', 'alice_bases', 'bob_bases', 'noise_level', 'noise_type'
    sift_a = run_result['sifted_alice']
    sift_b = run_result['sifted_bob']
    q = run_result['error_rate']
    # compute per-basis qber on *sifted* positions: need original bases mapped to indices
    # run_result should include sifted_indices to map back to alice/bob bases
    indices = run_result.get('sifted_indices', [])
    if indices:
        # build per-basis lists
        a_bases = [run_result['alice_bases'][i] for i in indices]
        b_bases = [run_result['bob_bases'][i] for i in indices]
        # here bases match by definition after sifting, so use a_bases for basis label
        z_idx = [i for i,b in enumerate(a_bases) if b==0]
        x_idx = [i for i,b in enumerate(a_bases) if b==1]
        def rate_from_indices(idx_list):
            if not idx_list: return 0.0
            mismatches = sum(sift_a[i]!=sift_b[i] for i in idx_list)
            return mismatches/len(idx_list)
        q_z = rate_from_indices(z_idx)
        q_x = rate_from_indices(x_idx)
    else:
        q_z, q_x = 0.0, 0.0

    # error vector on sifted bits
    err_vec = [1 if a!=b else 0 for a,b in zip(sift_a, sift_b)]
    bness = burstiness(err_vec)
    autoc = error_autocorr(err_vec)

    features = {
        'qber': q,
        'qber_z': q_z,
        'qber_x': q_x,
        'block_length': len(sift_a),
        'burstiness': bness,
        'autocorr_lag1': autoc,
        # include the simulator noise params if present
        'noise_level': run_result.get('noise_level', 0.0),
        'noise_type': run_result.get('noise_type', 'none')
    }
    return features


In [7]:
# Try to import pyldpc; if it fails, define a fallback simple parity reconciliation
try:
    from pyldpc import make_ldpc, encode, decode, get_message
    HAVE_PYLDPC = True
except Exception as e:
    HAVE_PYLDPC = False
    print("pyldpc not available; using a simple parity-check fallback for LDPC-like baseline.")

def simple_one_way_parity(alice_key, bob_key):
    """
    Very simple one-way parity: Alice sends the parity of the whole block;
    Bob flips if parity mismatch. This is naive but provides a baseline.
    Returns: corrected_bob, leaked_bits (parity length)
    """
    if not alice_key:
        return bob_key.copy(), 0
    parity_a = sum(alice_key) % 2
    parity_b = sum(bob_key) % 2
    corrected = bob_key.copy()
    leaked = 1
    if parity_a != parity_b:
        # naive: flip first bit (not secure in real protocols, but OK for baseline)
        corrected[0] ^= 1
    return corrected, leaked

def ldpc_reconcile(alice_key, bob_key):
    """
    Try to run LDPC reconciliation if pyldpc is present. This is a simplified wrapper.
    Returns corrected_bob, leaked_bits, success_flag
    """
    if not HAVE_PYLDPC:
        corr, leaked = simple_one_way_parity(alice_key, bob_key)
        return corr, leaked, False

    # convert lists to numpy arrays of ints 0/1
    m = len(alice_key)
    if m == 0:
        return bob_key.copy(), 0, False

    # choose small LDPC matrix for demonstration, adjust (n, d_v, d_c)
    n = m
    d_v = 2
    d_c = 4
    try:
        H, G = make_ldpc(n, d_v, d_c, systematic=True)
        # encode Alice's message
        msg = np.array(alice_key, dtype=int)
        codeword = encode(G, msg, snr=3)  # snr param is placeholder
        # Bob's noisy observation — convert bob_key to codeword-like vector
        y = np.array(bob_key, dtype=int)
        # decode
        x_hat = decode(H, y, maxiter=50)
        message = get_message(G, x_hat)
        corrected = [int(b) for b in message[:m]]
        leaked = H.shape[0]  # syndrome length - rough leakage
        success = (corrected == alice_key)
        return corrected, leaked, success
    except Exception as e:
        # fallback if pyldpc usage fails
        corr, leaked = simple_one_way_parity(alice_key, bob_key)
        return corr, leaked, False


pyldpc not available; using a simple parity-check fallback for LDPC-like baseline.


In [ ]:
import csv

def run_single_run(n=128, noise_level=0.02, noise_type='depolarizing', with_eve=False):
    # call your run_bb84 but ensure it returns the noise params in the dict
    res = run_bb84(n=n, with_eve=with_eve, verbose=False, print_circuits=False,
                   noise_level=noise_level, noise_type=noise_type, cascade=False, privacy_amp=False)
    # attach noise params so extract_features can pick them
    res['noise_level'] = noise_level
    res['noise_type'] = noise_type
    return res

def eval_ecs_on_run(run_result):
    """
    Run different EC algorithms and compute metrics.
    Returns dict with metrics for each algorithm and the 'best' algorithm label.
    """
    alice = run_result['sifted_alice']
    bob = run_result['sifted_bob']
    base_metrics = {}
    # Cascade
    a_c, b_c, corrections = cascade_error_correction(alice, bob, num_passes=4)
    err_after = error_rate(a_c, b_c)
    # compute simple secure_key_rate proxy: (len - leaked)/2 * (1 - 2*QBER) (toy)
    leaked_cascade = corrections  # rough proxy: each flip inferred leaks 1 bit
    final_len_c = len(a_c) - leaked_cascade
    secure_rate_c = max(final_len_c, 0) * max(0.0, 1 - 2*err_after)  # toy metric
    base_metrics['cascade'] = {
        'err_after': err_after,
        'leaked': leaked_cascade,
        'secure_metric': secure_rate_c,
        'corrected_key': a_c
    }

    # LDPC / one-way
    corr_ldpc, leaked_ldpc, success = ldpc_reconcile(alice, bob)
    err_after_ldpc = error_rate(alice, corr_ldpc)
    final_len_l = len(alice) - leaked_ldpc
    secure_rate_l = max(final_len_l, 0) * max(0.0, 1 - 2*err_after_ldpc)
    base_metrics['ldpc'] = {
        'err_after': err_after_ldpc,
        'leaked': leaked_ldpc,
        'secure_metric': secure_rate_l,
        'success': success
    }

    # choose best by secure_metric
    best = max(base_metrics.items(), key=lambda kv: kv[1]['secure_metric'])[0]
    return base_metrics, best

def run_grid_and_collect(save_csv='bb84_dataset.csv',
                         n_trials_per_point=20,
                         block_lengths=[64, 128, 256],
                         noise_levels=[0.0, 0.01, 0.02, 0.05, 0.1],
                         noise_types=['depolarizing', 'bitflip', 'phaseflip']):
    """
    Run many simulations and save to CSV: each row = features + best_algo label + metrics
    """
    fieldnames = ['qber','qber_z','qber_x','block_length','burstiness','autocorr_lag1',
                  'noise_level','noise_type',
                  # labels / metrics
                  'best_algo','cascade_secure_metric','ldpc_secure_metric',
                  'cascade_err_after','ldpc_err_after','cascade_leaked','ldpc_leaked']
    with open(save_csv, 'w', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        total = len(block_lengths)*len(noise_levels)*len(noise_types)*n_trials_per_point
        pbar = trange(total, desc='Sim runs')
        for bl in block_lengths:
            for nl in noise_levels:
                for nt in noise_types:
                    for t in range(n_trials_per_point):
                        res = run_single_run(n=bl, noise_level=nl, noise_type=nt)
                        features = extract_features(res)
                        metrics, best = eval_ecs_on_run(res)
                        row = {
                            **features,
                            'best_algo': best,
                            'cascade_secure_metric': metrics['cascade']['secure_metric'],
                            'ldpc_secure_metric': metrics['ldpc']['secure_metric'],
                            'cascade_err_after': metrics['cascade']['err_after'],
                            'ldpc_err_after': metrics['ldpc']['err_after'],
                            'cascade_leaked': metrics['cascade']['leaked'],
                            'ldpc_leaked': metrics['ldpc']['leaked']
                        }
                        writer.writerow(row)
                        pbar.update(1)
        pbar.close()
    print("Saved dataset to", save_csv)


ModuleNotFoundError: No module named 'tqdm'

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

def train_classifier_from_csv(csv_path='bb84_dataset.csv'):
    import pandas as pd
    df = pd.read_csv(csv_path)
    # simple preprocessing: encode noise_type
    enc = OneHotEncoder(sparse=False, handle_unknown='ignore')
    noise_type_encoded = enc.fit_transform(df[['noise_type']])
    # numeric features
    X_num = df[['qber','qber_z','qber_x','block_length','burstiness','autocorr_lag1','noise_level']].values
    X = np.hstack([X_num, noise_type_encoded])
    y = df['best_algo'].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    clf = RandomForestClassifier(n_estimators=200, random_state=42)
    clf.fit(X_train, y_train)
    acc = clf.score(X_test, y_test)
    print("Classifier test accuracy:", acc)
    return clf, enc

# Example:
# clf, enc = train_classifier_from_csv('bb84_dataset.csv')
